# Pilot B (standalone): quality-stratified conformal prediction on PTB-XL

Runs on CPU or GPU. Order: run all cells. Check the printed `smoke:` flag says False before trusting any number. Simulated noise only; not a paper result.

**Known simplifications:** noise is injected synthetically; folds 1-8 train, 9 calibration, 10 test (verify against the PTB-XL README); single-superclass records only.

In [ ]:
!pip install -q wfdb
import os, sys, copy, math, time, glob, subprocess, itertools
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

# ---- CONFIG ----
SMOKE = False        # True = synthetic data to test code only (never a result). False = real PTB-XL run.
QUICK_B = True       # True = 1 seed, 1 noise draw, 8 epochs (about 10 min on GPU). False = full run (3 seeds x 3 draws).
USE_DRIVE = False    # True = keep the PTB-XL download on Google Drive so a runtime restart does not lose it.
ALPHA = 0.1
SEEDS = [0, 1, 2]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PTBXL_ROOT = "/content/ptbxl"
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive"); PTBXL_ROOT = "/content/drive/MyDrive/ptbxl"
print("device:", DEVICE, "| smoke:", SMOKE, "| quick:", QUICK_B, "| data:", PTBXL_ROOT)
if DEVICE == "cpu": print("WARNING: CPU only. Use Runtime > Change runtime type > GPU, or expect it to be several times slower.")

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

## Download data

In [ ]:
# Download PTB-XL as ONE zip (much faster than crawling 20k small files), keep only metadata + 100 Hz records.
import shutil
def ensure_ptbxl(root=PTBXL_ROOT):
    csv = os.path.join(root, "ptbxl_database.csv")
    n_hea = lambda: len(glob.glob(os.path.join(root, "records100", "*", "*_lr.hea")))
    if os.path.exists(csv) and n_hea() >= 21000:
        print("PTB-XL already present:", n_hea(), "records"); return
    os.makedirs(root, exist_ok=True)
    URL = "https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip"
    zp = "/content/ptbxl.zip"
    print("downloading zip (about 1.8 GB; resumable, re-run this cell if it stops)...")
    subprocess.run(["wget", "-c", "-q", "--show-progress", "--tries=10", "--waitretry=3", "-O", zp, URL], check=True)
    tmp = "/content/ptbxl_tmp"; shutil.rmtree(tmp, ignore_errors=True)
    subprocess.run(["unzip", "-q", "-o", zp, "*/ptbxl_database.csv", "*/scp_statements.csv", "*/records100/*", "-d", tmp], check=True)
    src = glob.glob(tmp + "/*")[0]
    subprocess.run(["cp", "-a", src + "/.", root + "/"], check=True)   # merges, no nesting
    shutil.rmtree(tmp, ignore_errors=True)
    print("done:", n_hea(), "records100 files")
if not SMOKE: ensure_ptbxl()

## Code

In [ ]:
# %%
def lac_all(probs):
    """Score of every label: 1 - p(label). probs: (n, K)."""
    return 1.0 - probs

def aps_all(probs):
    """Deterministic APS score of every label: total prob mass of labels ranked at or above it."""
    order = np.argsort(-probs, axis=1)
    cum = np.cumsum(np.take_along_axis(probs, order, axis=1), axis=1)
    ranks = np.argsort(order, axis=1)
    return np.take_along_axis(cum, ranks, axis=1)

SCORES = {"LAC": lac_all, "APS": aps_all}

def conformal_q(true_scores, alpha=ALPHA):
    """Finite-sample split-conformal quantile. Returns inf if the calibration set is too small."""
    n = len(true_scores)
    k = int(math.ceil((n + 1) * (1 - alpha)))
    if n == 0 or k > n:
        return np.inf
    return float(np.sort(true_scores)[k - 1])

def summarize(sets, y, min_class_n=10):
    covered = sets[np.arange(len(y)), y]
    cls = [covered[y == c].mean() for c in np.unique(y) if (y == c).sum() >= min_class_n]
    return dict(coverage=float(covered.mean()), size=float(sets.sum(1).mean()),
                worst_class_cov=float(min(cls)) if cls else np.nan)


In [ ]:
# %%
class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.b2 = nn.BatchNorm2d(cout)
        self.sc = None
        if stride != 1 or cin != cout:
            self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        o = F.relu(self.b1(self.c1(x))); o = self.b2(self.c2(o))
        return F.relu(o + (x if self.sc is None else self.sc(x)))

class ResNet18Small(nn.Module):
    """ResNet-18 layout (2,2,2,2 blocks) with a 3x3 stem, suited to 28x28 images."""
    def __init__(self, num_classes, width=64):
        super().__init__()
        w = width
        self.stem = nn.Sequential(nn.Conv2d(3, w, 3, 1, 1, bias=False), nn.BatchNorm2d(w), nn.ReLU())
        cfg, layers, cin = [(w, 1), (w, 1), (2*w, 2), (2*w, 1), (4*w, 2), (4*w, 1), (8*w, 2), (8*w, 1)], [], w
        for cout, s in cfg:
            layers.append(Block(cin, cout, s)); cin = cout
        self.layers = nn.Sequential(*layers); self.fc = nn.Linear(cin, num_classes)
    def forward(self, x):
        x = self.layers(self.stem(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

class ECGNet(nn.Module):
    """Small 1D CNN for 12-lead ECG, input (N, 12, T)."""
    def __init__(self, num_classes, ch=(32, 64, 96, 128)):
        super().__init__()
        layers, cin = [], 12
        for c in ch:
            layers += [nn.Conv1d(cin, c, 7, 2, 3, bias=False), nn.BatchNorm1d(c), nn.ReLU()]; cin = c
        self.body = nn.Sequential(*layers); self.fc = nn.Linear(cin, num_classes)
    def forward(self, x):
        return self.fc(self.body(x).mean(-1))

def train_model(model, X, y, epochs, lr=2e-3, bs=256, seed=0):
    set_seed(seed); model.to(DEVICE).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * math.ceil(len(y) / bs))
    Xt, yt = torch.as_tensor(X), torch.as_tensor(y).long()
    for ep in range(epochs):
        perm = torch.randperm(len(yt))
        for i in range(0, len(yt), bs):
            idx = perm[i:i + bs]
            xb, yb = Xt[idx].to(DEVICE), yt[idx].to(DEVICE)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model.eval()

@torch.no_grad()
def predict_probs(model, X, bs=512):
    model.to(DEVICE).eval(); out = []
    Xt = torch.as_tensor(X)
    for i in range(0, len(Xt), bs):
        out.append(F.softmax(model(Xt[i:i + bs].to(DEVICE)), dim=1).cpu())
    return torch.cat(out).numpy().astype(np.float64)

In [ ]:
CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

def load_ptbxl(smoke, root=None):
    root = root or PTBXL_ROOT
    if smoke:
        rng = np.random.RandomState(1); K = 5; T = 1000
        def make(n):
            y = rng.randint(0, K, n); t = np.arange(T) / 100.0
            X = np.zeros((n, 12, T), np.float32)
            for i in range(n):
                f = 1.0 + 0.35 * y[i]  # class-dependent "heart rate"
                base = np.sin(2 * np.pi * f * t) + 0.5 * np.sin(2 * np.pi * 2 * f * t)
                X[i] = base[None] * rng.uniform(0.7, 1.3, (12, 1)) + 0.15 * rng.randn(12, T)
            return X, y
        (Xtr, ytr), (Xca, yca), (Xte, yte) = make(2500), make(800), make(800)
        return Xtr, ytr, Xca, yca, Xte, yte, K
    import ast, wfdb
    ensure_ptbxl(root)
    df = pd.read_csv(os.path.join(root, "ptbxl_database.csv"), index_col="ecg_id")
    df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)
    agg = pd.read_csv(os.path.join(root, "scp_statements.csv"), index_col=0)
    agg = agg[agg["diagnostic"] == 1]
    sup = lambda codes: sorted({agg.loc[c, "diagnostic_class"] for c in codes if c in agg.index})
    df["sc"] = df["scp_codes"].apply(sup)
    df = df[df["sc"].apply(len) == 1].copy()                       # single-superclass records only
    df["label"] = df["sc"].apply(lambda s: CLASSES.index(s[0]))
    print("records kept:", len(df), "| class counts:", df["label"].value_counts().sort_index().tolist())
    for col in ["static_noise", "burst_noise", "baseline_drift", "electrodes_problems"]:
        if col in df.columns:
            print(f"metadata '{col}': {df[col].notna().sum()} annotated records")  # check if usable as real quality labels
    ok = df["filename_lr"].apply(lambda f: os.path.exists(os.path.join(root, f + ".hea")) and os.path.exists(os.path.join(root, f + ".dat")))
    print("records with missing files (dropped):", int((~ok).sum()))
    df = df[ok]
    assert len(df) > 10000, "too many missing files: re-run the cell to resume the download"
    X = np.stack([wfdb.rdsamp(os.path.join(root, f))[0].T for f in df["filename_lr"]]).astype(np.float32)
    fold, y = df["strat_fold"].values, df["label"].values
    mu, sd = X[fold <= 8].mean((0, 2), keepdims=True), X[fold <= 8].std((0, 2), keepdims=True) + 1e-6
    X = (X - mu) / sd
    sel = lambda m: (X[m], y[m])
    (Xtr, ytr), (Xca, yca), (Xte, yte) = sel(fold <= 8), sel(fold == 9), sel(fold == 10)
    return Xtr, ytr, Xca, yca, Xte, yte, len(CLASSES)


In [ ]:
NOISE_LEVELS = [np.inf, 12.0, 6.0, 0.0]   # SNR in dB; inf = clean
LEVEL_NAMES = ["clean", "12dB", "6dB", "0dB"]

def add_noise(X, snr_db, rng, fs=100):
    """Add white / baseline-wander / muscle-like noise at a target SNR (per record)."""
    if not np.isfinite(snr_db): return X
    N, L, T = X.shape; t = np.arange(T) / fs
    kind = rng.randint(0, 3, N)
    n = np.zeros_like(X)
    for i in range(N):
        if kind[i] == 0:
            n[i] = rng.randn(L, T)
        elif kind[i] == 1:
            w = sum(np.sin(2 * np.pi * rng.uniform(0.05, 0.5) * t + rng.uniform(0, 6.28)) for _ in range(3))
            n[i] = w[None] * rng.uniform(0.5, 1.5, (L, 1))
        else:
            n[i] = np.diff(rng.randn(L, T + 1), axis=1)  # emphasises high frequencies
    n = n / (n.std(axis=(1, 2), keepdims=True) + 1e-9)
    sp = X.var(axis=(1, 2), keepdims=True)
    return (X + n * np.sqrt(sp / (10 ** (snr_db / 10)))).astype(np.float32)

def make_mixed(X, rng, probs=(0.4, 0.2, 0.2, 0.2)):
    lv = rng.choice(len(NOISE_LEVELS), size=len(X), p=probs)
    Xn = X.copy()
    for k, snr in enumerate(NOISE_LEVELS):
        m = lv == k
        if m.any(): Xn[m] = add_noise(X[m], snr, rng)
    return Xn, lv

def noise_index(X, fs=100):
    """Label-free quality proxy from the signal itself: high-frequency and low-frequency energy ratios."""
    v = X.var(axis=2) + 1e-9                                  # (N, L)
    hf = (np.diff(X, axis=2).var(axis=2) / (2 * v)).mean(1)
    k = fs
    lf = np.stack([np.apply_along_axis(lambda s: np.convolve(s, np.ones(k) / k, "same"), 1, X[i]).var(axis=1) / v[i]
                   for i in range(len(X))]).mean(1)
    return hf, lf

def groupwise_q(scores_true, groups, alpha=ALPHA):
    return {g: conformal_q(scores_true[groups == g], alpha) for g in np.unique(groups)}

def pilot_B(smoke=SMOKE, seeds=SEEDS, n_noise_draws=3, epochs=None):
    Xtr, ytr, Xca, yca, Xte, yte, K = load_ptbxl(smoke)
    epochs = epochs or (4 if smoke else 15)
    rows, count_rows = [], []
    for seed in seeds:
        t0 = time.time()
        model = train_model(ECGNet(K), Xtr, ytr, epochs, lr=3e-3, bs=128, seed=seed)
        print(f"seed {seed}: trained in {time.time()-t0:.0f}s (clean test acc {np.mean(predict_probs(model, Xte).argmax(1)==yte):.3f})")
        for d in range(n_noise_draws):
            rng = np.random.RandomState(7919 * seed + d)
            Xca_n, lca = make_mixed(Xca, rng); Xte_n, lte = make_mixed(Xte, rng)
            pca, pte = predict_probs(model, Xca_n), predict_probs(model, Xte_n)
            hf_ca, lf_ca = noise_index(Xca_n); hf_te, lf_te = noise_index(Xte_n)
            z = lambda a, ref: (a - ref.mean()) / (ref.std() + 1e-9)
            ni_ca = z(hf_ca, hf_ca) + z(lf_ca, lf_ca); ni_te = z(hf_te, hf_ca) + z(lf_te, lf_ca)
            edges = np.quantile(ni_ca, [0.25, 0.5, 0.75])
            bca, bte = np.digitize(ni_ca, edges), np.digitize(ni_te, edges)      # proxy quality bins (0=cleanest)
            clean_ca = lca == 0
            for sname, sfn in SCORES.items():
                Sca, Ste = sfn(pca), sfn(pte)
                tca = Sca[np.arange(len(yca)), yca]
                q_glob = conformal_q(tca)
                q_orc, q_prx = groupwise_q(tca, lca), groupwise_q(tca, bca)
                q_clean = conformal_q(tca[clean_ca])
                methods = {
                    "naive_mixed_cal":  np.full(len(yte), q_glob),
                    "naive_clean_cal":  np.full(len(yte), q_clean),
                    "mondrian_oracle":  np.array([q_orc[g] for g in lte]),
                    "mondrian_proxy":   np.array([q_prx[g] for g in bte]),
                }
                for mname, qv in methods.items():
                    sets = Ste <= qv[:, None]
                    for k, ln in enumerate(LEVEL_NAMES):
                        m = lte == k
                        r = summarize(sets[m], yte[m], min_class_n=5)
                        rows.append(dict(seed=seed, draw=d, score=sname, method=mname, level=ln, **r))
                if sname == "LAC" and d == 0:
                    count_rows.append(dict(seed=seed, **{f"cal_n_{ln}": int((lca == k).sum()) for k, ln in enumerate(LEVEL_NAMES)}))
    print("calibration counts per noise level (seed 0 draw 0):", count_rows[0] if count_rows else "n/a")
    return pd.DataFrame(rows)

def report_B(df):
    if SMOKE: print("*** SMOKE = True: SYNTHETIC DATA, NOT A RESULT ***")
    per = df.groupby(["score", "method", "level"])[["coverage", "size"]].mean().reset_index()
    per["level"] = pd.Categorical(per["level"], LEVEL_NAMES, ordered=True)
    cov = per.pivot_table(index=["score", "method"], columns="level", values="coverage", observed=True).round(3)
    siz = per.pivot_table(index=["score", "method"], columns="level", values="size", observed=True).round(2)
    print("COVERAGE by true noise level (target %.2f):" % (1 - ALPHA)); print(cov.to_string())
    print("\nMEAN SET SIZE by true noise level:"); print(siz.to_string())
    nv, mo = cov.loc[("APS", "naive_mixed_cal")], cov.loc[("APS", "mondrian_oracle")]
    mp = cov.loc[("APS", "mondrian_proxy")]
    under = (1 - ALPHA) - nv.min()
    print("\nPILOT B DECISION RULE (APS shown; check LAC too):")
    print(f"  naive worst-level undercoverage  = {100*under:.1f} pts (continue needs >= 5)")
    print(f"  mondrian_oracle worst-level gap   = {100*((1-ALPHA)-mo.min()):.1f} pts (want <= 2)")
    print(f"  mondrian_proxy  worst-level gap   = {100*((1-ALPHA)-mp.min()):.1f} pts (label-free quality bins)")
    print(f"  noisiest-level set size: naive {siz.loc[('APS','naive_mixed_cal'),'0dB']:.2f} | oracle-Mondrian {siz.loc[('APS','mondrian_oracle'),'0dB']:.2f} | max possible {len(CLASSES)}")
    ok = under >= 0.05 and ((1 - ALPHA) - mo.min()) <= 0.02 and siz.loc[("APS", "mondrian_oracle"), "0dB"] < 0.9 * len(CLASSES)
    print("  -> CONTINUE topic B" if ok else "  -> weak or unusable effect; see tables before deciding")
    return cov, siz


## Run

In [ ]:
dfB = pilot_B(seeds=[0], n_noise_draws=1, epochs=8) if QUICK_B else pilot_B()
covB, sizB = report_B(dfB)
dfB.to_csv("pilotB_raw.csv", index=False)